# Rsec · Colab · Phase 01

Runs a real tiny CPU smoke test by default. GPU QLoRA, downloads, resume and export require explicit flags below. Enable a provider GPU only when available; free hardware is not guaranteed. No trained Rsec cybersecurity model is distributed. Never paste credentials into a cell.

This notebook pins a tested implementation commit. A newer commit must be reviewed and pinned explicitly. Full documentation: https://github.com/Snowwiy/Rsec


In [ ]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path

RSEC_REVISION = "44084096dc231b7726ca1490c5e65f69a01956f9"
ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
CHECKOUT = ROOT / "Rsec"
RUN_GPU_TRAINING = False
RUN_SMOKE = True
ALLOW_MODEL_DOWNLOAD = False
RUN_EVALUATION = False
RUN_EXPORT = False
RUN_GGUF = False
USE_GOOGLE_DRIVE = False
RESUME_CHECKPOINT = ""
PREVIOUS_ADAPTER = ""
LLAMA_CPP_CHECKOUT = ""
assert re.fullmatch(r"[0-9a-f]{40}", RSEC_REVISION)
assert not (RESUME_CHECKPOINT and PREVIOUS_ADAPTER), "Choose resume or a fresh incremental run"

In [ ]:
if CHECKOUT.exists():
    actual = subprocess.check_output(
        ["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True
    ).strip()
    assert actual == RSEC_REVISION, "Existing checkout has another revision; use a new directory"
    assert not subprocess.check_output(
        ["git", "-C", str(CHECKOUT), "status", "--porcelain"], text=True
    ).strip(), "Existing checkout has modifications; preserve them"
else:
    subprocess.check_call(["git", "clone", "https://github.com/Snowwiy/Rsec.git", str(CHECKOUT)])
    subprocess.check_call(["git", "-C", str(CHECKOUT), "checkout", "--detach", RSEC_REVISION])
os.chdir(CHECKOUT)
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["DO_NOT_TRACK"] = "1"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-e", "."])
subprocess.check_call([sys.executable, "-m", "rsec.cli", "doctor"])

## Install the appropriate ML stack

A provider's PyTorch/CUDA image must be verified rather than replaced blindly. The CPU path uses the tested official wheel. GPU profiles are candidates pending physical validation. Run doctor after installation and keep the report with artifacts.


In [ ]:
if RUN_GPU_TRAINING:
    import torch

    assert torch.cuda.is_available(), "No CUDA GPU detected; do not simulate GPU availability"
    assert torch.__version__.split("+")[0] == "2.8.0", (
        "Use a separately verified PyTorch 2.8.0 CUDA environment"
    )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements-train.txt"])
else:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "torch==2.8.0+cpu",
            "--index-url",
            "https://download.pytorch.org/whl/cpu",
        ]
    )
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-r", "requirements-cpu-test.lock"]
    )
subprocess.check_call([sys.executable, "-m", "rsec.cli", "doctor"])

In [ ]:
import uuid

SESSION = "notebook-" + uuid.uuid4().hex[:10]
ARTIFACT_ROOT = ROOT / "rsec-artifacts" / SESSION
if USE_GOOGLE_DRIVE:
    assert "google.colab" in sys.modules or Path("/content").exists(), (
        "Drive mounting is Colab-only"
    )
    from google.colab import drive

    drive.mount("/content/drive")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/Rsec") / SESSION
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=False)

## Model selection and reviewed data

Edit the model identifier AND immutable revision together after checking the license, architecture and tokenizer. The default is the preliminary Qwen reference. Synthetic examples validate plumbing; replace them with a separately reviewed corpus before phase 03 training. The test split and independent benchmarks remain reserved.


In [ ]:
import yaml

from rsec.config import load_training

settings = load_training(profile="colab_t4").model_dump()
MODEL_ID = settings["model"]
MODEL_REVISION = settings["revision"]
DATA_RELEASE = ARTIFACT_ROOT / "dataset"
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "rsec.cli",
        "data",
        "prepare",
        "--input",
        "data/samples/defensive.jsonl",
        "--output",
        str(DATA_RELEASE),
    ]
)
subprocess.check_call([sys.executable, "-m", "rsec.cli", "data", "validate", str(DATA_RELEASE)])
settings.update(
    model=MODEL_ID,
    revision=MODEL_REVISION,
    dataset=str(DATA_RELEASE),
    output=str(ARTIFACT_ROOT / "experiments"),
)
CONFIG = ARTIFACT_ROOT / "training.yaml"
CONFIG.write_text(yaml.safe_dump(settings))

In [ ]:
if RUN_SMOKE or RUN_GPU_TRAINING:
    from rsec.config import load_training
    from rsec.train import train

    config = load_training(CONFIG)
    changes = {"local_files_only": not ALLOW_MODEL_DOWNLOAD}
    if PREVIOUS_ADAPTER:
        changes["adapter"] = PREVIOUS_ADAPTER
    config = type(config).model_validate({**config.model_dump(), **changes})
    result = train(config, smoke=not RUN_GPU_TRAINING, resume=RESUME_CHECKPOINT or None)
    RUN_PATH = Path(result["path"])
    print(json.dumps(result, indent=2))
else:
    RUN_PATH = None
    print("Training is disabled; configuration and data have been prepared.")

## Resume and evaluate

Set RESUME_CHECKPOINT to a prior full local checkpoint, preserve matching data/config and rerun the training cell. Incremental training instead uses PREVIOUS_ADAPTER and a new reviewed dataset with replay examples. CPU smoke metrics do not establish cybersecurity capability.


In [ ]:
if RUN_EVALUATION:
    assert RUN_GPU_TRAINING and RUN_PATH is not None, (
        "Cybersecurity evaluation needs an actual selected base/adapter"
    )
    from huggingface_hub import snapshot_download

    from rsec.evaluate import evaluate
    from rsec.predict import predict

    local_base = snapshot_download(
        MODEL_ID,
        revision=MODEL_REVISION,
        local_dir=str(ARTIFACT_ROOT / "base-model"),
        local_files_only=not ALLOW_MODEL_DOWNLOAD,
        allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja"],
    )
    predictions = ARTIFACT_ROOT / "predictions.jsonl"
    predict(
        "benchmarks/suite.jsonl", local_base, str(predictions), adapter=str(RUN_PATH / "adapter")
    )
    evaluate(
        "benchmarks/suite.jsonl",
        predictions,
        ARTIFACT_ROOT / "evaluation.json",
        result["experiment"],
    )

## Export and persist artifacts

Adapter export is lightweight. A merged/GGUF export needs an unquantized base, a reviewed llama.cpp build, converter dependencies and sufficient RAM/storage. Export may need another machine. No Ollama/GGUF compatibility is implied by successful adapter training.


In [ ]:
if RUN_EXPORT:
    assert RUN_PATH is not None
    from rsec.export import export

    export(RUN_PATH / "adapter", ARTIFACT_ROOT / "export-adapter", format="adapter")
if RUN_GGUF:
    assert RUN_GPU_TRAINING and RUN_PATH is not None and LLAMA_CPP_CHECKOUT
    from rsec.export import export

    assert "local_base" in globals(), (
        "Acquire/review the local unquantized base in the evaluation cell first"
    )
    export(RUN_PATH / "adapter", ARTIFACT_ROOT / "merged", format="merged", base=local_base)
    export(
        ARTIFACT_ROOT / "merged",
        ARTIFACT_ROOT / "gguf",
        format="gguf",
        llama_cpp=LLAMA_CPP_CHECKOUT,
    )
print("Preserve artifacts before the provider session expires:", ARTIFACT_ROOT)

In [ ]:
DOWNLOAD_ARCHIVE = False
if DOWNLOAD_ARCHIVE:
    import shutil

    from google.colab import files

    archive = shutil.make_archive(str(ARTIFACT_ROOT), "zip", ARTIFACT_ROOT)
    files.download(archive)